# Problema Direto Transitório 

Este problema tem como objetivo implementar uma PINN para um problema direto transitório.

**Autor:** Edélio Gabriel Magalhães de Jesus

## Definição do problema

O primeiro problema que discutimos tratava-se de um problema apenas de valor de contorno — logo, não considerávamos nenhuma evolução temporal. Por isso o chamamos de estacionário.

Agora, vamos adicionar um novo aspecto: as **condições iniciais**. Elas fundamentam o caráter temporal do problema — a solução não é mais um estado de equilíbrio, mas uma função que evolui no tempo a partir de um estado inicial conhecido.

Para isso, trabalharemos com a **Equação de Burgers**.

---

### ``Condições iniciais e de contorno``

O problema é resolvido no domínio:

$$
x \in [-1,1], \qquad t \in [0,1]
$$

com viscosidade:

$$
\nu = \frac{0.01}{\pi}
$$

A condição inicial é dada por:

$$
u(x,0) = -\sin(\pi x)
$$

As condições de contorno de Dirichlet homogêneas são:

$$
u(-1,t)=0
$$

e

$$
u(1,t)=0
$$

para todo $t \in [0,1]$.

---

### `Requisitos teóricos`

#### Equação de Burgers

A equação de Burgers é uma equação diferencial parcial parabólica do tipo convecção-difusão (Equação 1), que descreve o transporte de uma grandeza física por dois mecanismos simultâneos: **convecção** — o transporte pelo próprio campo — e **difusão** — o espalhamento devido à viscosidade [[ref]](#eq-convec-difu). Ela aparece em diversas áreas da matemática aplicada e da física, como mecânica dos fluidos, acústica não linear, dinâmica de gases e fluxo de tráfego.

$$
u_t + c\cdot u_x = \mu\cdot u_{xx} \tag{1}
$$

O aspecto mais discutido da equação de Burgers é a competição entre esses dois mecanismos: a convecção tende a acentuar gradientes e formar **choques** — descontinuidades abruptas na solução — enquanto a difusão os suaviza. Para viscosidades pequenas, os choques dominam e a solução desenvolve frentes muito íngremes, o que torna o problema numericamente desafiador e um benchmark clássico para métodos numéricos e, mais recentemente, para PINNs [[ref]](#original-paper).

<div style="text-align: center;">
  <img 
    src="https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Airplane_vortex.jpg/1280px-Airplane_vortex.jpg" 
    alt="Ilustração da mecânica de fluidos"
    style="max-width: 400px; width: 100%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://pt.wikipedia.org/wiki/Mec%C3%A2nica_dos_fluidos' target='_blank'>
    Wikipedia — Mecânica dos fluidos
  </a>
</div>

Ela é expressa por:

$$
u_t + u \cdot u_x = \nu \cdot u_{xx} \tag{2}
$$

onde $u(x, t)$ é o campo de velocidade, $\nu > 0$ é a viscosidade cinemática, $u \cdot u_x$ é o termo de **convecção não linear** e $\nu \cdot u_{xx}$ é o termo de **difusão**. Note que, diferentemente da equação de advecção-difusão linear, o coeficiente de convecção aqui é a própria solução $u$ — o que torna o problema não linear.

---

## Aplicando a PINN

O código completo está localizado na pasta `scripts`, especificamente no arquivo `ex03_pinn_direct_transient.py`. Para facilitar a discussão, colocarei apenas trechos necessários para uma compreensão mais aprofundada.

---

A célula seguinte serve para:

- Recarregar automaticamente qualquer arquivo que for editado nos scripts
- Encontrar a pasta dos *scripts*, permitindo importar as funções criadas

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath("../scripts"))

import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

### **Importações necessárias**

In [2]:
import torch.nn as nn
import torch.optim as optim
import torch
import plotly.graph_objects as go
from geral_functions import PINN, sample_collocation_rectangular, sample_boundary_rectangular_transient
from plot_utils import plot_loss, plot_heatmaps, plot_points_transient, plot_profiles
from ex03_pinn_direct_transient import train_burgers, evaluate_burgers, numerical_solution_burgers


### **Parâmetros do problema**

Os valores dos parâmetros que envolvem a arquitetura da rede a amostragem foram inspirados na discussão presente artigo original de "Raissi et. al. ("**Data-driven solutions of nonlinear partial differential equations**"[[ref]](#original-paper)), em que uma PINN foi implementada para esse meesmo problema.

In [3]:
# Definição do local onde o código serpa executado. Por padrão, gpu
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {DEVICE}')

# Arquitetura da rede
N_INPUTS = 2
N_OUTPUTS = 1
N_HIDDEN = 16
N_LAYERS = 9
ACTIVATION = nn.Tanh

# Parâmetros do problema
X_LB, X_UB = -1.0, 1.0
T_LB, T_UB = 0.0, 1.0
NU = 0.01 / torch.pi

IC_FN = lambda x: -torch.sin(torch.pi * x)

BC_FNS = {
    'left':  lambda t: torch.zeros_like(t),
    'right': lambda t: torch.zeros_like(t),
}

# Parâmetros de amostragem
N_COLLOC = 5000
N_BC = 150
N_IC = 150

# Parâmetros do treinamento
W_IC = 1.0
W_BC = 1.0
W_PDE = 1.0
N_EPOCHS = 10000
LR = 1e-4


Usando: cpu


### **Instanciando o modelo**

In [4]:
model = PINN(N_INPUTS, N_OUTPUTS, N_HIDDEN, N_LAYERS, ACTIVATION)

### **Amostragem dos pontos**

In [5]:
X_COLLOC = sample_collocation_rectangular(N_COLLOC, [X_LB, T_LB], [X_UB, T_UB], DEVICE)

X_IC, U_IC, X_BC, U_BC = sample_boundary_rectangular_transient(
    N_IC, N_BC,
    X_LB, X_UB,
    T_LB, T_UB,
    IC_FN, BC_FNS, DEVICE
)

---

> Vale destacar que estamos usando uma amostragem aleatória, enquanto que os autores optaram por uma amostragem pelo método quase-aleatório *Latin Hypercube Sampling (LHS)* - que é conhecido conseguir alcançar uma cobertura mais uniforme de todo o domínio. 

> Agora, você pode se perguntar: quais são os impactos da amostragem no aprendizado do nosso modelo?

> Se ficou curioso, confira o *notebook* `07_sampling.ipynb`!

---

### **Instanciando o otimizador**

In [6]:
OPTIMIZER = torch.optim.Adam(model.parameters(), lr=LR)

### **Treinamento**

In [7]:
history = train_burgers(model, OPTIMIZER, X_COLLOC, X_IC, U_IC, X_BC, U_BC, NU, N_EPOCHS, W_IC, W_BC, W_PDE)

Epoch 00000 | Loss: 5.77e-01 | Loss IC: 5.57e-01 | Loss BC: 1.94e-02 | Loss PDE: 8.74e-06
Epoch 00100 | Loss: 5.26e-01 | Loss IC: 5.26e-01 | Loss BC: 2.95e-04 | Loss PDE: 8.85e-05
Epoch 00200 | Loss: 4.67e-01 | Loss IC: 4.47e-01 | Loss BC: 1.68e-02 | Loss PDE: 3.43e-03
Epoch 00300 | Loss: 4.24e-01 | Loss IC: 3.36e-01 | Loss BC: 8.17e-02 | Loss PDE: 6.83e-03
Epoch 00400 | Loss: 3.90e-01 | Loss IC: 2.91e-01 | Loss BC: 9.43e-02 | Loss PDE: 4.62e-03
Epoch 00500 | Loss: 3.26e-01 | Loss IC: 2.29e-01 | Loss BC: 9.06e-02 | Loss PDE: 5.46e-03
Epoch 00600 | Loss: 2.97e-01 | Loss IC: 2.07e-01 | Loss BC: 7.72e-02 | Loss PDE: 1.22e-02
Epoch 00700 | Loss: 2.74e-01 | Loss IC: 1.94e-01 | Loss BC: 5.89e-02 | Loss PDE: 2.06e-02
Epoch 00800 | Loss: 2.38e-01 | Loss IC: 1.71e-01 | Loss BC: 3.80e-02 | Loss PDE: 2.96e-02
Epoch 00900 | Loss: 2.09e-01 | Loss IC: 1.41e-01 | Loss BC: 3.00e-02 | Loss PDE: 3.84e-02
Epoch 01000 | Loss: 1.77e-01 | Loss IC: 1.14e-01 | Loss BC: 1.69e-02 | Loss PDE: 4.57e-02
Epoch 0110

### **Visualizando os resultados do treinamento**

Nossa função de perda ficou da seguinte forma:

$$
Loss = \omega_{PDE} * L_{PDE} + \omega_{BC} * L_{BC} + \omega_{CI} *  L_{CI}
$$

sendo, **BC** os dados da condiçaõ de contorno e **CI**, da condição inicial. Note que, diferentemente da metodologia do *paper* original, aqui separamos as perdas das condições de contorno e inicial, ao invés de tratarmos tudo como dados.

No código temos:

---
```python
[...]
    # perda física 
    residual = pde_residual_burguers(model, X_col, nu)
    loss_pde = torch.mean(residual ** 2)

    # perda CI
    U_ic_pred = model(X_ic)
    loss_ic = torch.mean((U_ic_pred - U_ic) ** 2)

    # perda CC
    U_cc_pred = model(X_bc)
    loss_bc = torch.mean((U_cc_pred - U_bc) ** 2)

    # perda total 
    loss = w_ic * loss_ic + w_bc * loss_bc + w_pde * loss_pde 

    return loss, loss_ic, loss_bc, loss_pde
```
---
    

Olhemos para nossa distribuição espaço-temporal.

In [8]:
plot_points_transient(X_COLLOC, X_BC, X_IC)

Note a característica distinta em relação ao problema estacionário: precisamos garantir uma cobertura temporal!

Aqui, trabalhamos com um caso simples unidimensional. Mas para problemas com mais dimensões, e governados por equações mais complexas, a previsão se torna mais difícil e custosas.

In [9]:
plot_loss(history)

### **Validando o modelo**

Após o treinamento, a PINN é comparada com uma solução numérica de referência da equação de Burgers:

$$
u_t + u\,u_x = \nu u_{xx}
$$

Primeiramente, a solução de referência é obtida numericamente em uma malha de pontos no espaço e no tempo, utilizando diferenças finitas para aproximar as derivadas da equação.

A condição inicial utilizada é:

$$
u(x,0)=-\sin(\pi x)
$$

com condições de contorno que definimos:

$$
u(-1,t)=u(1,t)=0
$$

O termo convectivo \(u\,u_x\) é escrito na forma conservativa:

$$
\left(\frac{u^2}{2}\right)_x
$$

e discretizado com um fluxo numérico do tipo Local Lax–Friedrichs (Rusanov). Já o termo difusivo é aproximado por diferenças centrais. A evolução temporal é resolvida com o método implícito BDF (`solve_ivp`).

Depois disso, a PINN é avaliada nos mesmos pontos da malha utilizada pela solução numérica. Para cada par \((x,t)\), a rede neural retorna uma aproximação:

$$
u_\theta(x,t)
$$

A solução prevista pela PINN é então comparada com a solução de referência através do erro relativo em norma \(L_2\):

$$
\text{Erro}_{L_2}
=
\frac{\|U_{\text{pred}}-U_{\text{ref}}\|_2}
{\|U_{\text{ref}}\|_2}
$$

Além do erro global, também são comparados perfis da solução em diferentes instantes de tempo, permitindo verificar visualmente a qualidade da aproximação aprendida pela PINN.

---

#### ``/Pequena discussão sobre a conversão para a forma conservativa``



Na equação de Burgers, o termo não linear pode ser escrito como:

$$
u\,u_x
$$

ou, de forma equivalente:

$$
\left(\frac{u^2}{2}\right)_x
$$

No código, foi utilizada a segunda forma, chamada de **forma conservativa**.

Isso permite escrever a equação como:

$$
u_t + f(u)_x = \nu u_{xx}
$$

onde:

$$
f(u)=\frac{u^2}{2}
$$

A vantagem dessa escrita é que a variação da solução passa a ser calculada em termos de **fluxos** entre pontos vizinhos da malha.

Assim, o método numérico pode calcular quanto da quantidade \(u\) “entra” e “sai” de cada região do domínio, utilizando fluxos numéricos apropriados.

No código, isso possibilita utilizar o esquema de Local Lax–Friedrichs (Rusanov) para discretizar o termo convectivo de forma mais estável.

> Para mais detalhes, ver [[ref]](#burgers-eq).

---

Vamos gerar a solução numérica para o nosso problema:

In [10]:
x_ref, t_ref, U_ref = numerical_solution_burgers()

Agora, podemos avaliá-lo:

In [12]:
results = evaluate_burgers(
    model=model,
    x_ref=x_ref,
    t_ref=t_ref,
    U_ref=U_ref,
    device=DEVICE
)

Por fim, vamos visualizar os resultados.

In [13]:
plot_heatmaps(
    results['U_pred'], results['U_ref'],
    results['t'], results['x'],
    title='Equação de Burgers',
    xlabel='t', ylabel='x'
)

In [14]:
# snapshots temporais
plot_profiles(
    results['U_pred_snaps'], results['U_ref_snaps'],
    results['x'],
    slices=results['snap_times'],
    title='Snapshots temporais',
    xlabel='x', ylabel='u(x, t)',
    slice_label='t'
)

Pelos gráficos, podemos notar que o modelo aprendeu muito bem qualitativamente o nosso problema, identificando o choque em $x=0$. Note, no entando, que o erro nessa região ainda foi consideravelemnte alto: da ordem de $10^-1$.

Isso pode ser solucionado aumentando a complexidade da nossa rede, seja em profundidade (número de camadas), ou em largura (número de neurônios).


---

>
> Retornamos esse mesmo problema no *notebook* `arquitectures`, onde aplicamos uma PINN híbrida com outra arquitetura. Se ficou curioso, confira!

---

## Referências

<a id="original-paper"></a>  
RAISSI, Maziar; PERDIKARIS, Paris; KARNIADAKIS, George Em. *Physics informed deep learning (part i): Data-driven solutions of nonlinear partial differential equations*. arXiv preprint arXiv:1711.10561, 2017. Disponível em: https://arxiv.org/abs/1711.10561

<a id="burgers-eq"></a>  
UNIVERSIDADE FEDERAL DO RIO GRANDE DO SUL (UFRGS). *Equação de Burgers*. Disponível em: https://fiscomp.if.ufrgs.br/index.php/Equa%C3%A7%C3%A3o_de_Burgers#M%C3%A9todo_Conservativo

<a id="eq-comvec-difu"></a>  
WIKIPEDIA. *Equação de convecção-difusão*. Disponível em: https://pt.wikipedia.org/wiki/Equa%C3%A7%C3%A3o_de_convec%C3%A7%C3%A3o-difus%C3%A3o